In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np
 
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import timm

from dataset import PlantWildDataset
from dotenv import load_dotenv

In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

2.10.0+cu128
True
NVIDIA GeForce RTX 3060 Laptop GPU


In [ ]:
class MobileViT(nn.Module):
    """
    Full MobileViT fine-tune with all layers trainable.
    """
 
    def __init__(self, num_classes: int,
                 model_name: str, dropout: float = 0.2):
        super().__init__()
 
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        self.embed_dim = self.backbone.num_features
 
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, num_classes),
        )

        for param in self.backbone.parameters():
            param.requires_grad = True
 
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model           : {model_name}")
        print(f"Total params    : {total / 1e6:.2f}M")
        print(f"Trainable params: {trainable / 1e6:.2f}M  "
              f"({100 * trainable / total:.1f}% unfrozen)")
 
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(images))
 
    def get_encoder(self):
        return self.backbone
 
    def get_param_groups(self, backbone_lr: float, head_lr: float):
        """
        Separate LRs for backbone and head.
        Backbone gets a lower LR to preserve pretrained weights.
        """
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr},
            {"params": self.head.parameters(),     "lr": head_lr},
        ]

In [7]:
def get_transforms(img_size: int = 320, train: bool = True):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

In [ ]:
def train(model, train_loader, test_loader,
          epochs, backbone_lr, head_lr, device, save_dir,
          class_weights=None):
 
    os.makedirs(save_dir, exist_ok=True)
 
    model = model.to(device)
 
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=0.1,
    )
 
    optimizer = torch.optim.AdamW(
        model.get_param_groups(backbone_lr, head_lr),
        weight_decay=1e-4,
    )
 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )
 
    scaler   = GradScaler("cuda")
    best_acc = 0.0
 
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
 
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
 
            optimizer.zero_grad()
            with autocast("cuda"):
                loss = criterion(model(images), labels)
 
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
 
            train_loss += loss.item()
 
        scheduler.step(epoch)  
 
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast("cuda"):
                    preds = model(images).argmax(1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
 
        acc = 100 * correct / total
 
        print(f"Epoch {epoch:>3}/{epochs}  "
              f"Loss: {train_loss / len(train_loader):.4f}  "
              f"Test Acc: {acc:.2f}%")
 
        if acc > best_acc:
            best_acc = acc
            torch.save(
                model.get_encoder().state_dict(),
                os.path.join(save_dir, "best_image_encoder.pt")
            )
            print(f"  ✓ Best encoder saved ({best_acc:.2f}%)")
 
    print(f"\nFinetuning complete — best Test Acc: {best_acc:.2f}%")
    return model

In [ ]:
MODEL_NAME = "mobilevitv2_150"  
IMAGES_DIR  = "./data/images/plantwild"
IMG_SIZE    = 320      
BATCH_SIZE  = 16        
EPOCHS      = 50     
BACKBONE_LR = 1e-5      
HEAD_LR     = 1e-3      
SAVE_DIR  = f"./checkpoints"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
torch.cuda.manual_seed(42)
torch.manual_seed(42)

train_ds = PlantWildDataset(IMAGES_DIR,
                            transform=get_transforms(IMG_SIZE, train=True),
                            split="train")
test_ds  = PlantWildDataset(IMAGES_DIR,
                            transform=get_transforms(IMG_SIZE, train=False),
                            split="test", label_map=train_ds.label_map)

train_ds.save_label_map("./data/label_map.json")

class_counts  = train_ds.get_class_counts()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
print(f"Class weights computed for {len(class_weights)} classes")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True)

model = MobileViT(
    num_classes=len(train_ds.classes),
    model_name=MODEL_NAME,
    dropout=0.2,
)

model = train(
    model, train_loader, test_loader,
    epochs=EPOCHS,
    backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR,
    device=DEVICE,
    save_dir=SAVE_DIR,
    class_weights=class_weights,
)

In [ ]:
# load encoder
encoder = timm.create_model("mobilevitv2_150", pretrained=False, num_classes=0)
encoder.load_state_dict(
    torch.load("./checkpoints/best_image_encoder.pt",
               map_location=DEVICE)
)
encoder = encoder.to(DEVICE)
encoder.eval()

print(f"Encoder loaded — embed_dim: {encoder.num_features}")

In [ ]:
def extract_embeddings(encoder, dataloader, device):
    encoder.eval()
    all_embeddings = []
    all_labels     = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            with autocast("cuda"):
                embeddings = encoder(images)        
            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels)

    all_embeddings = torch.cat(all_embeddings, dim=0)   
    all_labels     = torch.cat(all_labels,     dim=0)   
    return all_embeddings, all_labels

train_ds_eval = PlantWildDataset(IMAGES_DIR,
                                  transform=get_transforms(IMG_SIZE, train=False),
                                  split="train",
                                  label_map=train_ds.label_map)
train_loader_eval = DataLoader(train_ds_eval, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=4,
                                pin_memory=True, persistent_workers=True)

print("Extracting train embeddings...")
train_embeddings, train_labels = extract_embeddings(encoder, train_loader_eval, DEVICE)

print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(encoder, test_loader, DEVICE)

print(f"Train embeddings : {train_embeddings.shape}")   
print(f"Train labels     : {train_labels.shape}")       
print(f"Test embeddings  : {test_embeddings.shape}")    
print(f"Test labels      : {test_labels.shape}")        

torch.save({
    "train_embeddings": train_embeddings,
    "train_labels":     train_labels,
    "test_embeddings":  test_embeddings,
    "test_labels":      test_labels,
}, "./checkpoints/image_embeddings.pt")

print("Saved → ./checkpoints/image_embeddings.pt")

PlantWildDataset | split=train | 89 classes | 14832 images
Extracting train embeddings...
Extracting test embeddings...
Train embeddings : torch.Size([14832, 768])
Train labels     : torch.Size([14832])
Test embeddings  : torch.Size([3708, 768])
Test labels      : torch.Size([3708])
Saved → ./checkpoints/image_embeddings.pt


In [15]:
data = torch.load("./checkpoints/image_embeddings.pt")
train_embeddings = data["train_embeddings"]   # (14832, 768)
train_labels     = data["train_labels"]       # (14832,)
test_embeddings  = data["test_embeddings"]    # (3708, 768)
test_labels      = data["test_labels"]        # (3708,)

train_embeddings

tensor([[ 0.7129, -0.3269,  0.0112,  ..., -0.0051, -0.4780,  0.0045],
        [ 0.1740, -0.1631,  0.0111,  ..., -0.0055, -0.1908, -0.4009],
        [ 0.5122,  0.1903,  0.0112,  ..., -0.0065, -0.1470, -0.1726],
        ...,
        [-0.1743,  0.0115,  0.0111,  ..., -0.0064,  0.1290,  0.2035],
        [ 0.2456, -0.4988,  0.0112,  ..., -0.0060,  0.0455,  0.0895],
        [-0.1276,  0.0447,  0.0112,  ..., -0.0072, -0.3223,  0.2340]],
       dtype=torch.float16)